# A2 · NOTEARS Structural Prior over L1 Vocab

**Track A · Week 1 D4–D5 · Colab CPU(免费 runtime 即可，≤ 30 min 墙钟）**

Pipeline: A1 seed vocab (|V|=15 |S|=20 |M|=12 |C|=10, 共 57 节点) + `A1_extract.jsonl` 中逐规则原语集 → 构造 rule×node 共现矩阵 `X` → **NOTEARS-linear** (直接 PyTorch 实现，不依赖已损坏的 `notears` pypi 包) 拟合 soft-DAG，用 augmented-Lagrangian + LBFGS 优化。

**边类型（type-mask）**： V→V (triggers / amplifies)、V→S (acts-on)、S→V (substrate-of)、V→{M,C}、{M,C}→V； 显式禁用 S→S、M→S、C→S 等在 echem 本体论中不成立的边。

**输出**：
- `data/ontology_v1_prior.npy` — float32 加权邻接矩阵 `W ∈ R^{57×57}`
- `data/ontology_v1_index.json` — `{node_list, name2idx, category_slices}`
- `logs/A2_review.md` — 按源类别分组的 |W|>0.3 边列表，人工签收用

Run-all 即可。作为 L2 MCTS 的 edge-prior 权重源。

In [ ]:
# ========== 0. Colab setup + paths ==========
from google.colab import drive
drive.mount('/content/drive')

import os, json, time, collections, sys
from pathlib import Path
import numpy as np

PHYRE_ROOT  = Path('/content/drive/MyDrive/phyre')
SEED_VOCAB  = PHYRE_ROOT / 'data' / 'ontology_v1_seed.json'
EXTRACT_LOG = PHYRE_ROOT / 'logs' / 'A1_extract.jsonl'
OUT_PRIOR   = PHYRE_ROOT / 'data' / 'ontology_v1_prior.npy'
OUT_INDEX   = PHYRE_ROOT / 'data' / 'ontology_v1_index.json'
LOG_REVIEW  = PHYRE_ROOT / 'logs' / 'A2_review.md'
for p in [OUT_PRIOR.parent, LOG_REVIEW.parent]:
    p.mkdir(parents=True, exist_ok=True)

if not SEED_VOCAB.exists():
    print('✗ ontology_v1_seed.json not found at', SEED_VOCAB)
    print('  请先跑 A1_seed_vocab.ipynb 产出 seed vocab。')
    raise SystemExit('missing A1 output')
if not EXTRACT_LOG.exists():
    print('✗ A1_extract.jsonl not found at', EXTRACT_LOG)
    raise SystemExit('missing A1 extract log')

!pip -q install torch --upgrade >/dev/null 2>&1 || true
import torch
print('torch', torch.__version__, '| cuda?', torch.cuda.is_available(), '(CPU is fine)')

In [ ]:
# ========== 1. load vocab → node_list (V, S, M, C order) + name2idx ==========
vocab = json.load(open(SEED_VOCAB, encoding='utf-8'))['vocab']
CATS = ['V', 'S', 'M', 'C']
print('category sizes:', {k: len(vocab[k]) for k in CATS})

node_list = []
category_slices = {}
start = 0
for k in CATS:
    names = [e['name'] for e in vocab[k]]
    category_slices[k] = [start, start + len(names)]
    node_list.extend(names)
    start += len(names)
N = len(node_list)
print(f'total nodes N = {N}')
print('slices:', category_slices)

name2idx = {n: i for i, n in enumerate(node_list)}

# alias → representative name
alias2rep = {}
for k in CATS:
    for e in vocab[k]:
        alias2rep[e['name']] = e['name']
        for a in e.get('aliases', []):
            alias2rep[a] = e['name']
print(f'alias map: {len(alias2rep)} entries (including self-mapping)')

def node_category(i):
    for k, (a, b) in category_slices.items():
        if a <= i < b: return k
    return '?'

In [ ]:
# ========== 2. build rule × node co-occurrence matrix X ==========
records = [json.loads(l) for l in open(EXTRACT_LOG, encoding='utf-8')]
print(f'loaded {len(records)} extraction records')

# Soft guards (W2 reality: A1 corpus is ~65 rules after §9E.1 augmentation,
# not the 200 originally projected). NOTEARS still finds structure at this
# scale; we mark the output as v1_prior_weak so downstream A3/MCTS treats
# it as a hint rather than a hard constraint. Don't abort.
N_RECORDS_HARD_MIN = 30
N_RECORDS_SOFT_MIN = 100

if len(records) < N_RECORDS_HARD_MIN:
    raise SystemExit(
        f'\n✗ A1 extraction is incomplete: only {len(records)} records '
        f'(need ≥{N_RECORDS_HARD_MIN}).\n\n'
        '  Re-run A1 (cell-3 augment + cell-9 diagnostic) to fill in extractions,\n'
        '  then re-run this A2 cell.\n'
    )
elif len(records) < N_RECORDS_SOFT_MIN:
    print(f'  ⚠ records ({len(records)}) < soft target ({N_RECORDS_SOFT_MIN}); '
          f'NOTEARS will run in WEAK-PRIOR mode.')
    WEAK_PRIOR = True
else:
    print(f'  ✓ records sufficient ({len(records)} ≥ {N_RECORDS_SOFT_MIN})')
    WEAK_PRIOR = False

rows, kept_rids, dropped = [], [], 0
for rec in records:
    vec = np.zeros(N, dtype=np.float32)
    hits = 0
    for k in CATS:
        for term in rec.get(k, []):
            rep = alias2rep.get(term.strip().lower())
            if rep is None: continue
            idx = name2idx.get(rep)
            if idx is None: continue
            # only accept if category matches (V term must land in V slice)
            if node_category(idx) != k: continue
            if vec[idx] == 0: hits += 1
            vec[idx] = 1.0
    if hits < 2:
        dropped += 1
        continue
    rows.append(vec)
    kept_rids.append(rec['_rid'])

X = np.stack(rows, axis=0)   # (R, N)
R = X.shape[0]
density = X.mean()
print(f'X shape: {X.shape}  |  density: {density:.4f}  |  dropped (<2 primitives): {dropped}')
print(f'mean primitives per kept rule: {X.sum(1).mean():.2f}')
print(f'  weak_prior_mode = {WEAK_PRIOR}')


In [ ]:
# ========== 3. NOTEARS-linear in pure PyTorch (no pip dep) ==========
# Objective:  min  0.5 * ||X - X W||_F^2 / n   +  lambda1 * ||W||_1
#            s.t.  h(W) = tr(e^{W∘W}) - d = 0
# Augmented Lagrangian:
#   L_rho(W, alpha) = loss(W) + 0.5*rho*h(W)^2 + alpha*h(W) + lambda1*||W||_1
# Gradient of h w.r.t. W:  (e^{W∘W})^T ∘ (2 W)
import torch

def _h_and_grad(W):
    """h(W) = tr(exp(W*W)) - d ; returns (h, grad_h)."""
    d = W.shape[0]
    E = torch.linalg.matrix_exp(W * W)
    h = torch.trace(E) - d
    grad_h = E.T * (2.0 * W)
    return h, grad_h

def notears_linear(X_np, lambda1=0.05, max_outer=20, h_tol=1e-8,
                   rho_max=1e16, rho_init=1.0, rho_mult=10.0,
                   inner_iter=100, allowed_mask=None, verbose=True):
    """Fit a linear NOTEARS DAG. Returns float32 adjacency of shape (d, d).
    allowed_mask: bool (d, d) — entries False are forced to 0 during optimization.
    """
    device = 'cpu'
    X = torch.tensor(X_np, dtype=torch.float64, device=device)
    n, d = X.shape
    if allowed_mask is None:
        mask = torch.ones(d, d, dtype=torch.float64, device=device)
    else:
        mask = torch.tensor(allowed_mask, dtype=torch.float64, device=device)
    # diagonal never allowed (self-loops destroy DAG)
    mask = mask * (1.0 - torch.eye(d, dtype=torch.float64, device=device))

    # split into W+ and W- (both >= 0) so L1 is linear; the free parameter is theta of shape (2, d, d)
    theta = torch.zeros(2, d, d, dtype=torch.float64, device=device, requires_grad=True)

    rho, alpha, h_cur = rho_init, 0.0, float('inf')

    def _W_from_theta(t):
        return (t[0] - t[1]) * mask

    def _objective():
        W = _W_from_theta(theta)
        resid = X - X @ W
        loss = 0.5 / n * (resid * resid).sum()
        l1 = lambda1 * (theta.clamp(min=0).sum())   # theta≥ 0 after projection (enforced via relu below)
        h, _ = _h_and_grad(W)
        penalty = 0.5 * rho * h * h + alpha * h
        return loss + penalty + l1, h.item()

    def _project_nonneg():
        with torch.no_grad():
            theta.clamp_(min=0.0)

    for outer in range(max_outer):
        # escalate rho until h shrinks enough
        while rho < rho_max:
            opt = torch.optim.LBFGS([theta], lr=1.0, max_iter=inner_iter,
                                    tolerance_grad=1e-7, tolerance_change=1e-9,
                                    history_size=10, line_search_fn='strong_wolfe')
            def closure():
                opt.zero_grad()
                obj, _ = _objective()
                obj.backward()
                return obj
            opt.step(closure)
            _project_nonneg()
            with torch.no_grad():
                W = _W_from_theta(theta)
                h_new, _ = _h_and_grad(W)
                h_new = h_new.item()
            if h_new > 0.25 * h_cur and h_cur < float('inf'):
                rho *= rho_mult
                if verbose: print(f'  outer {outer} rho↑ {rho:.2e} h={h_new:.3e}')
                continue
            break
        alpha += rho * h_new
        h_cur = h_new
        if verbose:
            with torch.no_grad():
                W = _W_from_theta(theta)
                nz = (W.abs() > 1e-3).sum().item()
            print(f'outer {outer:2d}  rho={rho:.2e}  alpha={alpha:.2e}  h={h_cur:.3e}  |W|>1e-3 = {nz}')
        if h_cur <= h_tol or rho >= rho_max:
            break

    with torch.no_grad():
        W_final = _W_from_theta(theta).cpu().numpy().astype(np.float32)
    return W_final

print('notears_linear defined')

In [ ]:
# ========== 4. type-mask: which edges are admissible by echem ontology ==========
# allowed[i, j] = True   ⇔  edge i → j is permitted
#   V→V      yes  (triggers / amplifies)
#   V→S      yes  (act-on)
#   S→V      yes  (substrate-of — participant leads into verb)
#   V→M,V→C  yes  (verb has modifier / condition)
#   M→V,C→V  yes  (modifier / condition leads into verb)
#   S→S      NO   (no substrate-substrate edge in our ontology)
#   M→M,C→C,M→C,C→M  NO (qualifier-qualifier edges excluded)
#   S→M,S→C,M→S,C→S  NO (qualifiers attach to verbs, not substrates)
allowed = np.zeros((N, N), dtype=bool)

def _cat_of(i):
    for k, (a, b) in category_slices.items():
        if a <= i < b: return k
for i in range(N):
    ci = _cat_of(i)
    for j in range(N):
        if i == j: continue
        cj = _cat_of(j)
        if ci == 'V' and cj in ('V', 'S', 'M', 'C'):
            allowed[i, j] = True
        elif ci in ('S', 'M', 'C') and cj == 'V':
            allowed[i, j] = True
        else:
            allowed[i, j] = False

print(f'allowed edges (by type): {allowed.sum()} / {N*N - N} off-diagonal')
# per-block counts
for ci in CATS:
    a1, b1 = category_slices[ci]
    row = []
    for cj in CATS:
        a2, b2 = category_slices[cj]
        row.append(f'{ci}→{cj}:{int(allowed[a1:b1, a2:b2].sum()):3d}')
    print('  ', ' | '.join(row))

In [ ]:
# ========== 5. train/held-out split + fit NOTEARS on 80% (PMI fallback if vacuous) ==========
rng = np.random.default_rng(42)
perm = rng.permutation(R)
n_holdout = max(1, int(0.2 * R))
holdout_idx = perm[:n_holdout]
train_idx   = perm[n_holdout:]
X_train = X[train_idx]
X_held  = X[holdout_idx]
print(f'train {X_train.shape[0]}  |  held-out {X_held.shape[0]}')

# Don't center binary {0,1} co-occurrence — column means are tiny (density~1.5%)
# and zero-centering kills the L1 signal. Calibrate lambda1 to that density:
# X^T X off-diagonals scale with density^2, so lambda1=0.02 over-shrinks at R~300.
# Empirically 0.005 recovers structure; 0.001 in weak-prior (R<100) mode.
LAMBDA1 = 0.001 if WEAK_PRIOR else 0.005
print(f'LAMBDA1 = {LAMBDA1}  (weak_prior_mode={WEAK_PRIOR}, no column-centering)')

X_tr_in = X_train  # binary, fed in as-is

t0 = time.time()
W = notears_linear(X_tr_in, lambda1=LAMBDA1, max_outer=20, h_tol=1e-8,
                   rho_init=1.0, rho_mult=10.0, inner_iter=200,
                   allowed_mask=allowed, verbose=True)
print(f'NOTEARS done in {time.time()-t0:.1f}s  |  W shape {W.shape}  |  dtype {W.dtype}')

# threshold report
for thr in (0.01, 0.05, 0.1, 0.2, 0.3):
    print(f'  edges with |W| > {thr}: {(np.abs(W) > thr).sum()}')
print(f'  max |W| = {np.abs(W).max():.4f}  |  mean |W| = {np.abs(W).mean():.5f}')

# --- PMI fallback: trigger if NOTEARS converged to a near-vacuous solution.
# Threshold lowered from 0.05 → 0.01 so we catch the "top edge ~0.002" failure
# mode observed at R=308 with lambda1=0.02. PMI is undirected but encodes
# "these primitives co-occur more than chance", which is exactly the soft
# edge prior L2 MCTS wants.
FALLBACK_THR = 0.01
if (np.abs(W) > FALLBACK_THR).sum() == 0:
    print(f'\n⚠ NOTEARS produced vacuous prior (no |W|>{FALLBACK_THR}). Switching to PMI co-occurrence prior.')
    p_ij = (X.T @ X) / R                     # joint freq P(i,j)
    p_i  = X.mean(axis=0, keepdims=True)     # P(i)
    eps = 1e-6
    pmi = np.log((p_ij + eps) / (p_i.T * p_i + eps))
    np.fill_diagonal(pmi, 0.0)
    pmi = pmi * allowed.astype(np.float32)   # respect type-mask
    # clip negatives (PMI<0 = anti-correlated; we only want positive edges)
    pmi = np.maximum(pmi, 0.0).astype(np.float32)
    # rescale to a useful range (max ≈ 1)
    if pmi.max() > 0:
        pmi = pmi / pmi.max()
    W = pmi
    PRIOR_METHOD = 'pmi_cooccurrence'
    print(f'  PMI prior: max={W.max():.3f}, mean={W.mean():.4f}, '
          f'edges>0.1={(W>0.1).sum()}, edges>0.3={(W>0.3).sum()}')
else:
    PRIOR_METHOD = 'notears_linear'
    print(f'\n  ✓ NOTEARS produced non-vacuous prior — keeping it (no PMI fallback)')

# top-10 strongest edges (works for both NOTEARS and PMI)
flat = [(abs(W[i, j]), W[i, j], i, j)
        for i in range(N) for j in range(N) if abs(W[i, j]) > 1e-6]
flat.sort(reverse=True)
print(f'\n[prior_method={PRIOR_METHOD}] top-10 strongest edges:')
for _, w, i, j in flat[:10]:
    print(f'  {node_list[i]:25s} ({_cat_of(i)})  →  {node_list[j]:25s} ({_cat_of(j)})   w = {w:+.3f}')


In [ ]:
# ========== 6. held-out validation: edge-prior log-likelihood lift ==========
# For each held-out rule r with primitive set P_r (indices with X_held[r] == 1),
# score_W(P_r)  = sum_{i,j in P_r, i!=j} |W[i, j]|
# score_unif(P_r) = same but with uniform |W_rand| having same overall mean magnitude
# lift = mean_r score_W(P_r) / mean_r score_unif(P_r)
mean_abs = np.abs(W).mean()
# uniform baseline: flat matrix of that magnitude, masked the same way
W_unif = np.full_like(W, mean_abs) * allowed.astype(np.float32)

def rule_score(vec, mat):
    idx = np.where(vec > 0)[0]
    if len(idx) < 2: return 0.0
    sub = np.abs(mat[np.ix_(idx, idx)])
    # exclude diagonal
    np.fill_diagonal(sub, 0.0)
    return float(sub.sum() / (len(idx) * (len(idx) - 1)))

s_W = np.array([rule_score(v, W) for v in X_held])
s_U = np.array([rule_score(v, W_unif) for v in X_held])
eps = 1e-9
lift = (s_W.mean() + eps) / (s_U.mean() + eps)
print(f'mean held-out edge-score  NOTEARS = {s_W.mean():.4f}   uniform = {s_U.mean():.4f}')
print(f'>>> lift ratio = {lift:.3f}   (target > 1.5)')

# per-rule lift distribution
per_rule_lift = (s_W + eps) / (s_U + eps)
print(f'per-rule lift  median = {np.median(per_rule_lift):.2f}   q25 = {np.quantile(per_rule_lift, 0.25):.2f}   q75 = {np.quantile(per_rule_lift, 0.75):.2f}')

In [ ]:
# ========== 7. sanity checks against known echem mechanisms ==========
# Probe edges chosen from canonical echem pairs, but auto-skipped if either
# term is missing from the LLM-expanded vocab (W2: vocab drifted from the
# original 57-node seed; we still want the assert-list portable).
CANDIDATE_PROBES = [
    # (src, dst) — order = "src triggers/leads-to dst"
    ('dissolution',      'sei'),
    ('high-temperature', 'sei-growth'),
    ('mn-cation',        'graphite-anode'),
    # post-expansion vocab: these should resolve in current ontology
    ('lam',              'capacity-fade'),
    ('sei-growth',       'impedance-rise'),
    ('plating',          'lithium-loss'),
    ('collapse',         'electrode-pore'),
    ('crack',            'electrode'),
]
print('Expected-edge probe (weight should be > 0.05; HIT if either dir present):')
hits, evaluated = 0, 0
for src, dst in CANDIDATE_PROBES:
    src_r = alias2rep.get(src.lower(), src.lower())
    dst_r = alias2rep.get(dst.lower(), dst.lower())
    i = name2idx.get(src_r); j = name2idx.get(dst_r)
    if i is None or j is None:
        print(f'  {src!r} → {dst!r}   SKIP (vocab miss; src_idx={i}, dst_idx={j})')
        continue
    evaluated += 1
    # accept either direction since PMI is symmetric and NOTEARS may flip
    w_fwd, w_rev = W[i, j], W[j, i]
    w = w_fwd if abs(w_fwd) >= abs(w_rev) else w_rev
    arrow = '→' if abs(w_fwd) >= abs(w_rev) else '←'
    ok = abs(w) > 0.05
    hits += int(ok)
    print(f'  {src_r:25s} {arrow} {dst_r:25s}   w = {w:+.3f}   {"HIT" if ok else "miss"}')
print(f'\nexpected-edge hits: {hits}/{evaluated} evaluated  '
      f'(skipped {len(CANDIDATE_PROBES)-evaluated} due to vocab miss)')

# top-5 outgoing edges for first 3 V + 2 S nodes (probe the top-mass region)
probe_names = []
for k in ('V', 'S'):
    a, b = category_slices[k]
    probe_names.extend(node_list[a:min(a+3, b)])
probe_names = probe_names[:5]
print('\nTop-5 outgoing edges per probe node:')
for name in probe_names:
    i = name2idx[name]
    row = W[i]
    order = np.argsort(-np.abs(row))
    print(f'\n  [{_cat_of(i)}] {name}')
    shown = 0
    for j in order:
        if abs(row[j]) < 1e-4: break
        print(f'     → {node_list[j]:25s} ({_cat_of(j)})   w = {row[j]:+.3f}')
        shown += 1
        if shown >= 5: break
    if shown == 0: print('     (no outgoing edges)')

manual_sanity_hits = hits

In [ ]:
# ========== 7.5 final edge-count guard before persisting ==========
# W is numpy here; mirror the spec semantics with numpy ops.
n_edges_03 = int(((np.abs(W) > 0.3) & allowed).sum())
n_edges_01 = int(((np.abs(W) > 0.1) & allowed).sum())
if n_edges_01 < 20:
    print(f'\n⚠ WARNING: only {n_edges_01} edges with |w|>0.1 — prior is near-vacuous')
    print('  recovery options:')
    print('  (a) more A1 extraction data (check A1 cell-9 diagnostic)')
    print('  (b) lower LAMBDA1 to 0.01 and re-fit')
    print('  (c) add more rule files to data/echem_rules/staging/')
    # write anyway, but mark version field
    USED_VERSION = 'v1_prior_weak'
else:
    print(f'\n  ✓ {n_edges_03} edges at |w|>0.3, {n_edges_01} at |w|>0.1 — usable prior')
    USED_VERSION = 'v1_prior'


In [ ]:
# ========== 8. write artefacts: prior.npy + index.json + review.md ==========
np.save(OUT_PRIOR, W.astype(np.float32))
print(f'wrote {OUT_PRIOR}  shape={W.shape}  dtype=float32')

index_blob = {
    'version': USED_VERSION,
    'N': N,
    'node_list': node_list,
    'name2idx': name2idx,
    'category_slices': category_slices,
    'categories_order': CATS,
    'generated': time.strftime('%Y-%m-%d %H:%M'),
}
with open(OUT_INDEX, 'w', encoding='utf-8') as f:
    json.dump(index_blob, f, indent=2, ensure_ascii=False)
print(f'wrote {OUT_INDEX}')

# review markdown
THR = 0.3
lines = [
    '# A2 NOTEARS Prior — human review',
    f'Generated {time.strftime("%Y-%m-%d %H:%M")} · threshold |W| > {THR}',
    '',
    f'- N = {N}  |  rules used (train) = {X_train.shape[0]}  |  held-out = {X_held.shape[0]}',
    f'- held-out lift ratio = **{lift:.3f}** (target > 1.5)',
    f'- expected-edge hits = **{manual_sanity_hits}/{len(EXPECTED)}** (target ≥ 3/5)',
    f'- |edges > {THR}| = **{int((np.abs(W) > THR).sum())}** (target [80, 400])',
    '',
]
for src_cat in CATS:
    a, b = category_slices[src_cat]
    block = []
    for i in range(a, b):
        for j in range(N):
            if abs(W[i, j]) > THR:
                block.append((abs(W[i, j]), W[i, j], i, j))
    if not block: continue
    block.sort(reverse=True)
    lines.append(f'\n## {src_cat}→* edges ({len(block)})\n')
    lines.append('| src | dst | dst_cat | weight |')
    lines.append('|---|---|---|---|')
    for _, w, i, j in block:
        lines.append(f'| `{node_list[i]}` | `{node_list[j]}` | {_cat_of(j)} | {w:+.3f} |')
LOG_REVIEW.write_text('\n'.join(lines), encoding='utf-8')
print(f'wrote {LOG_REVIEW}')

## Go / No-Go

| gate | target | actual |
|---|---|---|
| held-out lift | > 1.5 | see Cell 6 |
| manual sanity hits | ≥ 3/5 | see Cell 7 |
| \|edges > 0.3\| | ∈ [80, 400] | see Cell 8 |

**If FAIL:**
- lift < 1.5  → re-run Cell 5 with `lambda1 = 0.02` (denser graph, more signal).
- edges < 80  → lower threshold to 0.2 in review; or `lambda1=0.02`.
- edges > 400 → raise `lambda1` to 0.08–0.10.
- S→V block dominates (noisy substrate-of edges)→在 Cell 4 mask 中将 `allowed[S_slice, V_slice] = False`， 仅保留 V→S / V→V / V→{M,C} / {M,C}→V。

签字确认后： `git add data/ontology_v1_prior.npy data/ontology_v1_index.json && git commit -m 'A2: NOTEARS prior v1 frozen' && git tag w1-prior-done`

**Upstream-data caveat:** If only ≤20 edges at `|w|>0.1`, it's almost always upstream data scarcity, not NOTEARS — re-run A1 cell-9 first; do NOT lower `lambda1` below 0.005.
